# Run the HydroCNHS FE Walter DKP example / 執行 FE Walter DKP

## Goal / 目標
This repository runs hydrocnhs.Model.run, the Dam API, shared physics and downstream routing using frozen inflow. The DKP decision adapter adds retrieved knowledge to FE Walter public facts and the current daily state. The preserved builder retrieves relevant document sections; it does not paste the whole DKP or fine-tune a model. The actual system prompt and DKP remain separate files.

本 repo 用固定入流執行 hydrocnhs.Model.run、Dam API、共同物理及下游 routing。決策 adapter 將檢索知識加入 FE Walter 公開資料與當日狀態；原 builder 選取文件章節，不貼完整 DKP，也不 fine-tune 模型。System prompt 與 DKP 保持獨立。

## Setup / 準備
Python 3.12 with repository dependencies. Local files only; no provider calls. / 使用 Python 3.12 與既有依賴；使用局部檔案，不呼叫 provider。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
from tutorials.inspect_decision_flow import show_decision_flow
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: HydroCNHS-Delaware-Operations


## Steps

### Inspect the prompt and knowledge entry / 查看 prompt 與知識入口

In [2]:
print((ROOT/'examples/fe_walter/dkp/experiment.hydroopslm.offline.yaml').read_text())
print((ROOT/'prompts/fe_walter_reference/dkp_system.txt').read_text())
dkp=(ROOT/'data/anik_v3_7/DKP_FEWalter_Dam_v1.2.md').read_text()
print('DKP headings:', [s for s in dkp.splitlines() if s.startswith('#')][:12])

# HydroOpsLM offline teaching run; original settings/results are preserved.
# FE Walter V3.7 dkp: original prompt builders, synthetic replies, zero model calls.
schema_version: llm-reservoir-experiment.v1
case: ../../../cases/fe_walter_reference/case.json # Frozen hydrology sample; V3.7 physics.
run_id: hydroopslm-fe-walter-dkp        # Change with output before a new run.
output: ../../../runs/hydroopslm-fe-walter-dkp
max_calls: 0                             # This example cannot call a provider.
auditors:
  release_limits:
    type: release_bounds
    version: "1"
agents:
  fe_operator:
    reservoir: FEWalter                  # Reservoir ID in case.json.
    mode: offline                        # Native prompt/audit run on fixture replies.
    strategy: reference_v3_7             # Generic strategy selector; optional vendor plugin.
    reference:
      profile: fe_walter                 # This reference profile is FE-specific.
      method: dkp                   # Original zero-shot

### Run and inspect retrieval evidence / 執行並查看檢索證據

In [3]:
settings, verified=ensure_offline('examples/fe_walter/dkp/experiment.hydroopslm.offline.yaml')
print(json.dumps(verified,indent=2))
raw=[json.loads(p.read_text()) for p in (Path(settings.report['output'])/'raw').glob('policy_*.json')]
first=min(raw,key=lambda item:item['backend_trace']['input_variables']['timestep'])
prompt=first['backend_trace']['prompt']
assert 'Retrieved DKP sections' in prompt
assert dkp not in prompt
print('Retrieved excerpt, first 1200 characters:',prompt[prompt.index('Retrieved DKP sections'):][:1200])
print('Native audit retained:', 'backend_audit' in first)

{
  "status": "PASS",
  "days": 3,
  "reservoir_count": 1,
  "reservoir_days": 3,
  "date_start": "2016-01-01",
  "date_end": "2016-01-03",
  "audit_counts": {
    "PASS": 3
  },
  "max_reservoir_balance_residual_m3": 0.0,
  "max_system_balance_residual_m3": 0.0,
  "max_routing_difference_cms": 0.0,
  "final_channel_storage_m3": 0.0,
  "observed_network_calls": 0,
  "operating_strategy_accepted": null,
  "audit_binding_counts": {
    "versioned_suite": 3,
    "legacy_unbound": 0
  },
  "external_audit_pins_checked": []
}
Retrieved excerpt, first 1200 characters: Retrieved DKP sections relevant to current conditions:

--- DKP excerpt 1: R-RELEASE (composite per-timestep decision; WCM Table 7-1 Parts A and B; §7-01, §7-03, §7-05) ---
```pseudo
plan = R-PLAN(pool_elevation, stages, season)

active_min = R-MINFC       if plan in {C, D} or impounding_for_flood_emergency   # 100 cfs
           | R-MINDROUGHT  if drbc_drought_declared                              #  43 cfs
           | R-MINN

### Follow info, prompt, proposal, checker, audit/trace and feedback / 追查 info、prompt、proposal、checker、audit／trace 與 feedback

In [4]:
flow=show_decision_flow(Path(settings.report['output']))
assert flow['checker']['action_contract_recheck']=='PASS'
assert flow['next_info']['state']['storage_m3']==flow['execution']['storage_end_m3']
print('Original audit preserved:', flow['audit']['status'])


=== decision_id ===
86bf3ce7610c3cc9ec185dc885570f31a880a6cfbbe820211f878a05388b71b0

=== data_kind ===
synthetic_fixture

=== info ===
{
  "context": {
    "evaporation_m3": 0.0,
    "reservoir_id": "fe_walter",
    "seepage_m3": 0.0
  },
  "inflow_cms": 32.12531870666853,
  "release_upper_bound_cms": 60.572065702675154,
  "schema_version": "reservoir-observation.v1",
  "state": {
    "capacity_m3": 135532852.33895192,
    "dead_storage_m3": 49.3392742128,
    "dt_seconds": 86400.0,
    "maximum_release_cms": 283.16847,
    "minimum_release_cms": 1.41584235,
    "storage_m3": 2457848.279729185
  },
  "timestamp": "2016-01-01T00:00:00",
  "timestep": 0
}

=== prompt_variables ===
{
  "date": "2016-01-01",
  "history_semantics": "prior requested and implemented outflows; appended after physical commit",
  "inflow_cfs": 1134.4949071013637,
  "inflow_cms": 32.12531870666853,
  "native_history": [],
  "pool_elevation_start_ft": 1300.0,
  "reservoir_profile": "fe_walter",
  "schema_version

### Reject an invalid proposal before execution / 在執行前拒絕錯誤 proposal

In [5]:
from llm_reservoir_agent.contracts import parse_action
try:
    parse_action({'schema_version':'reservoir-action.v1','release_cms':-1})
except ValueError as error:
    print('Checker rejected negative release:', str(error))
else:
    raise AssertionError('invalid action accepted')
print('This isolated contract check calls no solver and changes no saved state.')

Checker rejected negative release: release_cms must be non-negative
This isolated contract check calls no solver and changes no saved state.


## Checks / 檢查
Review printed inputs, counts, sources and residuals. Synthetic replies validate interfaces only. / 核對輸入、計數、來源與殘差；合成回覆僅驗證介面。

## Next steps / 下一步
The native V3.7 DKP builder/auditor is FE-specific. Other reservoirs use the configured prompt/knowledge/auditor path with their own profile and rules; merely renaming FE Walter is rejected.

原 V3.7 builder／auditor 為 FE 專用；其他水庫使用 configured 與自己的 profile、prompt、knowledge、auditor；僅改名會拒絕。